# Radar de Startups Brasil
## Análise e Visualização de Dados com Python

**Disciplina:** Linguagem de Programação  
**Professor:** Louzada  
**Aluno:** Lucas de Almeida Brugger Cavalcante  
**Data:** Outubro de 2026

### Tema
Relação entre investimento, setor, tecnologia, localização e desempenho financeiro das startups brasileiras.


## 1. Introdução

O ecossistema de startups brasileiro apresenta empresas de diferentes setores, regiões, níveis de inovação e estágios de desenvolvimento. A análise desses dados permite identificar padrões que podem apoiar decisões relacionadas a investimento, expansão e priorização de mercados.

Neste projeto são analisados indicadores de investimento recebido, faturamento, crescimento, setores, regiões, estados, tecnologias e estágio das startups.


## 2. Problema de negócio

**Pergunta principal:** Como investimento, setor, tecnologia e localização se relacionam com o desempenho financeiro das startups brasileiras?

A resposta pode apoiar decisões sobre quais setores e regiões apresentam maior potencial, além de identificar características associadas a maior faturamento e crescimento.


## 3. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 4. Leitura da base de dados

O arquivo esperado é `simulacao_startups_brasil.csv`, localizado na pasta `dados`.


In [ ]:
from pathlib import Path

caminhos = [
    Path("dados/simulacao_startups_brasil.csv"),
    Path("simulacao_startups_brasil.csv")
]

arquivo = next((p for p in caminhos if p.exists()), None)

if arquivo is None:
    raise FileNotFoundError(
        "CSV não encontrado. Coloque simulacao_startups_brasil.csv na pasta dados."
    )

df = pd.read_csv(arquivo)
df.columns = [c.strip().lower() for c in df.columns]

print("Arquivo:", arquivo)
print("Linhas:", len(df))
print("Colunas:", len(df.columns))
df.head()


## 5. Conhecendo os dados

In [ ]:
display(df.head())
display(df.info())


In [ ]:
print("Dimensões:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

print("\nValores ausentes:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDuplicados:", df.duplicated().sum())


## 6. Tratamento e preparação dos dados

In [ ]:
# Conversão das principais variáveis numéricas, quando existentes
numericas = [
    "ano", "mes", "funcionarios", "investimento_recebido",
    "faturamento", "crescimento_percentual", "nivel_inovacao"
]

for col in numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Criação de uma variável temporal
if {"ano", "mes"}.issubset(df.columns):
    df["data"] = pd.to_datetime(
        dict(year=df["ano"], month=df["mes"], day=1),
        errors="coerce"
    )

print("Tratamento concluído.")
display(df.head())


## 7. Indicadores gerais (KPIs)

In [ ]:
def soma(col):
    return df[col].sum() if col in df.columns else np.nan

def media(col):
    return df[col].mean() if col in df.columns else np.nan

print(f"Investimento total: R$ {soma('investimento_recebido'):,.2f}")
print(f"Faturamento total: R$ {soma('faturamento'):,.2f}")
print(f"Crescimento médio: {media('crescimento_percentual'):.2f}%")
print(f"Registros analisados: {len(df):,}")

if "startup" in df.columns:
    print(f"Startups: {df['startup'].nunique():,}")


## 8. Análise por setor

A comparação dos setores permite identificar onde estão concentrados o faturamento e o investimento.


In [ ]:
if "setor" in df.columns:
    setor = df.groupby("setor", as_index=False).agg(
        faturamento=("faturamento", "sum") if "faturamento" in df.columns else ("setor", "size"),
        investimento=("investimento_recebido", "sum") if "investimento_recebido" in df.columns else ("setor", "size")
    ).sort_values("faturamento", ascending=False)

    display(setor.head(10))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=setor.head(10), x="faturamento", y="setor")
    plt.title("Top 10 setores por faturamento")
    plt.xlabel("Faturamento")
    plt.ylabel("Setor")
    plt.tight_layout()
    plt.show()


## 9. Análise por estado

A distribuição por UF ajuda a identificar regiões com maior concentração de atividade financeira das startups.


In [ ]:
if "uf" in df.columns:
    uf = df.groupby("uf", as_index=False).agg(
        faturamento=("faturamento", "sum") if "faturamento" in df.columns else ("uf", "size"),
        investimento=("investimento_recebido", "sum") if "investimento_recebido" in df.columns else ("uf", "size")
    ).sort_values("faturamento", ascending=False)

    display(uf.head(10))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=uf.head(10), x="faturamento", y="uf")
    plt.title("Top 10 UFs por faturamento")
    plt.xlabel("Faturamento")
    plt.ylabel("UF")
    plt.tight_layout()
    plt.show()


## 10. Evolução temporal

A análise temporal permite verificar tendências de crescimento, investimento e faturamento ao longo dos anos.


In [ ]:
if "ano" in df.columns and "faturamento" in df.columns:
    temporal = df.groupby("ano", as_index=False).agg(
        faturamento=("faturamento", "sum"),
        investimento=("investimento_recebido", "sum")
    )

    display(temporal)

    plt.figure(figsize=(11, 5))
    sns.lineplot(data=temporal, x="ano", y="faturamento", marker="o")
    plt.title("Evolução anual do faturamento")
    plt.xlabel("Ano")
    plt.ylabel("Faturamento")
    plt.tight_layout()
    plt.show()


## 11. Investimento x faturamento

In [ ]:
if {"investimento_recebido", "faturamento"}.issubset(df.columns):
    plt.figure(figsize=(9, 6))
    sns.scatterplot(
        data=df,
        x="investimento_recebido",
        y="faturamento",
        hue="setor" if "setor" in df.columns else None,
        alpha=0.65
    )
    plt.title("Relação entre investimento recebido e faturamento")
    plt.xlabel("Investimento recebido")
    plt.ylabel("Faturamento")
    plt.tight_layout()
    plt.show()


## 12. Correlação entre variáveis numéricas

A correlação ajuda a verificar relações lineares entre investimento, faturamento, crescimento, funcionários e inovação.


In [ ]:
cols_corr = [
    c for c in [
        "funcionarios",
        "investimento_recebido",
        "faturamento",
        "crescimento_percentual",
        "nivel_inovacao"
    ] if c in df.columns
]

if len(cols_corr) >= 2:
    corr = df[cols_corr].corr()
    display(corr)

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Matriz de correlação")
    plt.tight_layout()
    plt.show()


## 13. Tecnologia principal

In [ ]:
if "tecnologia_principal" in df.columns:
    tecnologia = df["tecnologia_principal"].value_counts().head(10)
    display(tecnologia)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=tecnologia.values, y=tecnologia.index)
    plt.title("Principais tecnologias utilizadas")
    plt.xlabel("Quantidade de registros")
    plt.ylabel("Tecnologia")
    plt.tight_layout()
    plt.show()


## 14. Estágio das startups

In [ ]:
if "estagio" in df.columns:
    estagio = df["estagio"].value_counts()
    display(estagio)

    plt.figure(figsize=(9, 5))
    sns.barplot(x=estagio.values, y=estagio.index)
    plt.title("Distribuição das startups por estágio")
    plt.xlabel("Quantidade")
    plt.ylabel("Estágio")
    plt.tight_layout()
    plt.show()


## 15. Principais insights

A análise permite observar:

- quais setores concentram maior faturamento;
- quais estados apresentam maior atividade financeira;
- como investimento e faturamento se relacionam;
- como o desempenho evolui ao longo do tempo;
- quais tecnologias aparecem com maior frequência;
- como as startups se distribuem entre os diferentes estágios.

Esses indicadores podem apoiar decisões de investimento, expansão regional e priorização de setores.


## 16. Conclusão

A análise demonstra como dados financeiros, temporais e características das startups podem ser combinados para apoiar decisões de negócio.

O dashboard desenvolvido em Streamlit complementa esta análise permitindo filtrar os dados por período, região, UF, setor e estágio, tornando a exploração dos resultados mais dinâmica.

Como conclusão executiva, os maiores potenciais devem ser avaliados considerando simultaneamente faturamento, investimento, crescimento, setor e localização, evitando decisões baseadas em apenas um indicador.
